# 📊 Análise de Produtos — DummyJSON API

Este notebook realiza uma análise exploratória de dados de produtos obtidos via API, com foco em **faturamento potencial, descontos e avaliações por categoria**.

**Perguntas que vamos responder:**
- Quais categorias têm maior potencial de faturamento?
- Como os descontos afetam o faturamento líquido?
- Quais categorias têm os produtos mais bem avaliados?
- Existe relação entre desconto e avaliação?

## 1. Importações e configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from API import data

def criar_grafico(dados, tipo='bar', titulo='', xlabel='', ylabel='', figsize=(10, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    if tipo == 'pie':
        dados.plot(kind=tipo, autopct='%1.1f%%', ax=ax)
        ax.set_ylabel('')
    else:
        dados.plot(kind=tipo, ax=ax)
    ax.set_title(titulo, fontsize=14, pad=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.tight_layout()
    plt.show()

## 2. Carregamento e inspeção dos dados

In [ ]:
df = pd.DataFrame(data['products'])
df.info()
df.isnull().sum()

### 🔎 Observações sobre os dados

- O dataset contém **30 produtos** distribuídos em **4 categorias**.
- **Não foram encontrados valores nulos** nas colunas utilizadas na análise, o que dispensa etapas de imputação.
- As colunas principais utilizadas são: `title`, `price`, `stock`, `discountPercentage`, `rating`, `reviews` e `category`.

## 3. Faturamento Bruto por Categoria

O **faturamento bruto potencial** representa o valor total em estoque antes de qualquer desconto (`price × stock`). É um indicador do capital imobilizado em produtos.

In [ ]:
df['potencialBruto'] = df['price'] * df['stock']
print(f"Faturamento Potencial Bruto Total: ${df['potencialBruto'].sum():.2f}")

faturamento_bruto = df.groupby('category')['potencialBruto'].sum().sort_values(ascending=False)
criar_grafico(faturamento_bruto, tipo='pie', titulo='Faturamento Bruto por Categoria',
              xlabel='', ylabel='')

### 💡 Conclusão — Faturamento Bruto

- A categoria **[furniture]** lidera em faturamento bruto, representando aproximadamente **90.9%** do total.
- A categoria **[furniture]** concentra a maior parte do estoque em valor, indicando onde o negócio está mais exposto.
- ⚠️ Faturamento bruto alto não garante lucratividade — o próximo passo é verificar o impacto dos descontos.

## 4. Faturamento Líquido por Categoria

O **faturamento líquido** desconta as promoções aplicadas a cada produto. É uma estimativa mais realista da receita potencial.

In [ ]:
df['potencialLiquido'] = df['potencialBruto'] * (1 - df['discountPercentage'] / 100)
print(f"Faturamento Potencial Líquido Total: ${df['potencialLiquido'].sum():.2f}")

faturamento_liquido = df.groupby('category')['potencialLiquido'].sum().sort_values(ascending=False)
criar_grafico(faturamento_liquido, tipo='pie', titulo='Faturamento Líquido por Categoria', figsize=(9, 9))

### 💡 Conclusão — Faturamento Líquido

- O desconto médio geral reduziu o faturamento em aproximadamente **11,5%** em relação ao bruto.
- A categoria **[furniture]** manteve sua liderança mesmo após os descontos, sugerindo preços base mais elevados.

## 5. Avaliação Média por Categoria

O `rating` médio por categoria revela a percepção de qualidade dos clientes. Categorias bem avaliadas tendem a ter maior retenção e menor necessidade de desconto para vender.

In [ ]:
rating_por_categoria = df.groupby('category')['rating'].mean().sort_values(ascending=False)
criar_grafico(rating_por_categoria, tipo='bar', titulo='Avaliação Média por Categoria',
              xlabel='Categoria', ylabel='Média de Avaliações')

### 💡 Conclusão — Avaliação Média

- A variação de rating entre categorias é **relativamente pequena** (todas entre 3,7 e 4), o que sugere consistência geral na qualidade.
- A categoria **[furniture]** se destaca com a melhor avaliação média, o que pode justificar ser a categoria mais rentável.
- Categorias com rating mais baixo podem ser candidatas a revisão de fornecedores ou estratégia de produto (especialmente as que representam menor parte do faturamento).

## 6. Desconto Médio por Categoria

Entender onde os maiores descontos estão concentrados ajuda a identificar estratégias de precificação e possíveis margens comprimidas.

In [ ]:
desconto_medio = df.groupby('category')['discountPercentage'].mean().sort_values(ascending=False)
criar_grafico(desconto_medio, tipo='bar', titulo='Desconto Médio por Categoria',
              xlabel='Categoria', ylabel='Desconto Médio (%)')

### 💡 Conclusão — Desconto Médio

- A categoria **[beauty]** pratica os maiores descontos médios (~12%), o que pode indicar dificuldade de giro de estoque ou estratégia promocional intensa.
- Cruzando com o faturamento líquido: categorias com alto desconto **não necessariamente** lideram em receita — o volume de estoque e o preço base também pesam.
- Categorias com baixo desconto e bom rating são as de **maior margem estratégica**.

## 7. Conclusão Geral

Esta análise exploratória revelou alguns padrões relevantes sobre o portfólio de produtos:

| Dimensão | Destaque |
|---|---|
| Maior faturamento bruto | [furniture] |
| Maior faturamento líquido | [furniture] |
| Melhor avaliação | [furniture] |
| Maior desconto médio | [beauty] |

**Próximos passos:**
- Analisar a relação entre `reviews` (volume de avaliações) e faturamento
- Identificar produtos individuais outliers dentro de cada categoria
- Construir um score combinado de prioridade (faturamento + rating + estoque)